# 08: Qwen3-8B 비추론 비증강 QLoRA 학습, Colab

07의 대응 노트북과 같은 모델, 프롬프트 및 파라미터를 사용하며 학습 증강 확률은 0.0입니다. `prefix_path`는 08의 데이터 디렉터리를 가리킵니다. 상단 딕셔너리를 확인한 뒤 순서대로 실행합니다. 다른 실험의 데이터와 어댑터를 읽지 않습니다.

07의 실제 실행과 비교할 때는 `train_desktop_unsloth.ipynb`로 학습하고 `evaluate_desktop.ipynb`로 평가합니다. 일반 학습 및 Colab 경로는 별도 실행 구성으로 보존하며 백엔드와 환경 차이를 증강 효과에 섞어 해석하지 않습니다. 자세한 실행 및 비교 조건은 [실험 안내](README.md)를 따릅니다.


## 1. 파라미터와 환경


In [ ]:
RUNTIME_PARAMS = {'prefix_path': '/drive/MyDrive/SiteBoard/08/',
 'model_name': 'Qwen/Qwen3-8B',
 'train_file': 'train.jsonl',
 'test_file': 'test.jsonl',
 'output_dir': 'models/checkpoints',
 'adapter_path': 'models',
 'run_config_file': 'result/run_config.json',
 'training_history_file': 'result/training_history.json',
 'report_file': 'result/evaluation.json',
 'device': 'cuda',
 'model_revision': 'b968826d9c46dd6066d109eabc6255188de91218'}

HYPER_PARAMS = {
    # 07과 같은 구현에서 확률 0.0으로 모든 문장 결합 증강을 끕니다.
    'augmentation': {
        'probability': 0.0,
        'max_concat': 2,
        'delimiters': [' ', '  ', ' / ', ' . ', '\n', ' , ', ' + '],
    },
    'chat_template': {'enable_thinking': False},
    'training': {
        'epochs': 3,
        'batch_size': 1,
        'grad_accum': 8,
        'learning_rate': 0.0002,
        'max_seq_length': 1280,
        'seed': 42,
    },
    'lora': {
        'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
        'task_type': 'CAUSAL_LM', 'bias': 'none',
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                           'gate_proj', 'up_proj', 'down_proj'],
    },
    'quantization': {
        'load_in_4bit': True,
        'bnb_4bit_quant_type': 'nf4',
        'bnb_4bit_use_double_quant': True,
    },
    # auto는 BF16 지원 GPU에서 BF16, 그 외에는 FP16을 선택합니다.
    'precision': {'compute_dtype': 'auto', 'frozen_embedding_dtype': 'auto'},
    'model_loading': {
        'device_map': {'': 0}, 'attn_implementation': 'sdpa', 'low_cpu_mem_usage': True,
    },
    'tokenizer': {'padding_side': 'right'},
    'kbit_training': {
        'use_gradient_checkpointing': True,
        'gradient_checkpointing_kwargs': {'use_reentrant': False},
    },
    'trainer': {
        'per_device_eval_batch_size': 1,
        'lr_scheduler_type': 'cosine', 'warmup_steps': 10,
        'logging_steps': 10, 'save_strategy': 'epoch', 'eval_strategy': 'epoch',
        'save_total_limit': 2, 'optim': 'paged_adamw_8bit',
        'dataloader_num_workers': 0, 'dataloader_pin_memory': True,
        'report_to': 'none', 'remove_unused_columns': False,
        'prediction_loss_only': True,
    },
    'collator': {'pad_to_multiple_of': 8, 'return_tensors': 'pt', 'padding': True},
}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 06과 같이 Colab 설치 셀에서 패키지를 설치합니다. 버전 범위는 루트 requirements.txt와 같습니다.
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", *['torch>=2.4.0', 'transformers>=4.57.6,<5', 'hf_xet>=1.1.3,<2', 'peft>=0.17.1,<0.18', 'accelerate>=0.29.0,<2', 'bitsandbytes>=0.50.2,<1', 'tqdm>=4.66.0', 'ipykernel>=6.29.0,<7', 'ipywidgets>=8,<9']])


## 2. 라이브러리


In [ ]:
import hashlib
import json
import sys
import time
import platform
import random
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, List

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)


## 3. 경로와 결과 저장


In [ ]:
def runtime_path(field):
    return str(Path(RUNTIME_PARAMS["prefix_path"]) / RUNTIME_PARAMS[field])


def write_json(path, value, mode="x"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open(mode, encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write("\n")


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_dtype(name):
    if name == "auto":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return getattr(torch, name)


## 4. 데이터 읽기


In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def prepare_training_data():
    train_records = read_jsonl(runtime_path("train_file"))
    valid_records = read_jsonl(runtime_path("test_file"))
    return train_records, valid_records, {
        "train_count": len(train_records), "test_count": len(valid_records),
    }


## 5. 원본 입력과 비추론 토큰화


In [ ]:
class QwenChatDataset:
    """같은 현장의 레코드를 랜덤 결합하고 JSON 응답 구간만 학습합니다."""

    def __init__(self, records, tokenizer, max_seq_length, is_train,
                 aug_prob, aug_max_concat, delimiters):
        self.records = records
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.is_train = is_train
        self.aug_prob = aug_prob
        self.aug_max_concat = aug_max_concat
        self.delimiters = delimiters
        self.site_to_indices = {}
        self.components = []
        for index, record in enumerate(records):
            self.site_to_indices.setdefault(record["metadata"]["site"], []).append(index)
            messages = record["messages"]
            rules, text = messages[1]["content"].split("[보드판 텍스트]\n", 1)
            self.components.append((messages[0]["content"], rules, text,
                                    json.loads(messages[2]["content"])))

    def __len__(self):
        return len(self.records)

    def encode_messages(self, messages):
        # 비추론 프롬프트와 JSON 응답을 분리해 경계 토큰까지 정확히 마스킹합니다.
        prompt = self.tokenizer.apply_chat_template(
            messages[:2], tokenize=False, add_generation_prompt=True,
            **HYPER_PARAMS["chat_template"],
        )
        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        response_ids = self.tokenizer.encode(messages[2]["content"], add_special_tokens=False)
        response_ids.append(self.tokenizer.eos_token_id)
        input_ids = prompt_ids + response_ids
        if len(input_ids) > self.max_seq_length:
            raise ValueError(f"학습 입력과 정답 {len(input_ids)}토큰이 상한 {self.max_seq_length}을 넘습니다. 잘라서 학습하지 않습니다.")
        return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
                "labels": [-100] * len(prompt_ids) + response_ids}

    def __getitem__(self, index):
        messages = self.records[index]["messages"]
        if self.is_train and random.random() < self.aug_prob:
            system, rules, text, answers = self.components[index]
            texts, combined = [text], list(answers)
            pool = self.site_to_indices[self.records[index]["metadata"]["site"]]
            for _ in range(random.randint(2, self.aug_max_concat) - 1):
                # 06과 같이 자기 자신도 포함한 동일 현장 풀에서 복원 추출합니다.
                partner = random.choice(pool)
                _, _, partner_text, partner_answers = self.components[partner]
                texts.append(partner_text)
                combined.extend(partner_answers)
            messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": rules + "[보드판 텍스트]\n" + random.choice(self.delimiters).join(texts)},
                {"role": "assistant", "content": json.dumps(combined, ensure_ascii=False, indent=2)},
            ]
        return self.encode_messages(messages)


def build_dataset(records, tokenizer, is_train):
    settings = HYPER_PARAMS["augmentation"]
    return QwenChatDataset(
        records, tokenizer, HYPER_PARAMS["training"]["max_seq_length"], is_train,
        settings["probability"], settings["max_concat"], settings["delimiters"],
    )




## 6. 모델과 학습기


In [ ]:
def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        RUNTIME_PARAMS["model_name"], revision=RUNTIME_PARAMS["model_revision"],
        **HYPER_PARAMS["tokenizer"],
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_qlora_model():
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])
    model = AutoModelForCausalLM.from_pretrained(
        RUNTIME_PARAMS["model_name"], revision=RUNTIME_PARAMS["model_revision"],
        dtype=dtype,
        quantization_config=BitsAndBytesConfig(
            bnb_4bit_compute_dtype=dtype, **HYPER_PARAMS["quantization"],
        ),
        **HYPER_PARAMS["model_loading"],
    )
    model.config.use_cache = False
    return model


def add_lora_adapter(model):
    model = prepare_model_for_kbit_training(model, **HYPER_PARAMS["kbit_training"])
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["frozen_embedding_dtype"])
    # PEFT가 FP32로 올린 고정 임베딩과 출력 계층만 16비트로 되돌립니다.
    for module in (model.get_input_embeddings(), model.get_output_embeddings()):
        if module is not None:
            module.to(dtype=dtype)
    model = get_peft_model(model, LoraConfig(**HYPER_PARAMS["lora"]))
    model.print_trainable_parameters()
    return model


def build_trainer(model, tokenizer, train_records, valid_records):
    settings = HYPER_PARAMS["training"]
    train_dataset = build_dataset(train_records, tokenizer, is_train=True)
    valid_dataset = build_dataset(valid_records, tokenizer, is_train=False)
    dtype = resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])
    args = TrainingArguments(
        output_dir=runtime_path("output_dir"), num_train_epochs=settings["epochs"],
        per_device_train_batch_size=settings["batch_size"],
        gradient_accumulation_steps=settings["grad_accum"],
        learning_rate=settings["learning_rate"], seed=settings["seed"],
        bf16=dtype == torch.bfloat16, fp16=dtype == torch.float16,
        **HYPER_PARAMS["trainer"],
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_dataset, eval_dataset=valid_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, **HYPER_PARAMS["collator"]),
    )
    return trainer


## 7. 실행 정보와 학습


In [ ]:
def collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(), "python_executable": sys.executable,
        "platform": platform.platform(), "packages": packages,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or RUNTIME_PARAMS["model_revision"],
        "compute_dtype": str(resolve_dtype(HYPER_PARAMS["precision"]["compute_dtype"])),
        "input_embedding_dtype": str(model.get_input_embeddings().weight.dtype) if model else None,
        "output_embedding_dtype": str(model.get_output_embeddings().weight.dtype) if model else None,
        "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
        "peak_allocated_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "peak_reserved_bytes": torch.cuda.max_memory_reserved() if torch.cuda.is_available() else None,
    }

def run_training():
    for key in ("run_config_file", "training_history_file", "report_file"):
        if Path(runtime_path(key)).exists():
            raise FileExistsError(f"기존 산출물이 있습니다: {runtime_path(key)}")
    adapter = Path(runtime_path("adapter_path"))
    if adapter.exists() and any(p.name != "README.md" for p in adapter.iterdir()):
        raise FileExistsError(f"기존 모델 또는 체크포인트가 있습니다: {adapter}")
    set_seed(HYPER_PARAMS["training"]["seed"])
    train_records, valid_records, prepared_data = prepare_training_data()
    tokenizer = load_tokenizer()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = add_lora_adapter(load_qlora_model())
    trainer = build_trainer(model, tokenizer, train_records, valid_records)
    paths = [runtime_path(key) for key in
             ("train_file", "test_file")]
    run_config = {
        "runtime_params": deepcopy(RUNTIME_PARAMS), "hyper_params": deepcopy(HYPER_PARAMS),
        "prepared_data": prepared_data, "prepared_file_hashes": {path: file_sha256(path) for path in paths},
        "execution": collect_execution_metadata(model), "status": "training_started",
    }
    write_json(runtime_path("run_config_file"), run_config)
    trainer.train()
    destination = Path(runtime_path("adapter_path"))
    model.save_pretrained(destination)
    tokenizer.save_pretrained(destination)
    write_json(runtime_path("training_history_file"), trainer.state.log_history)
    run_config.update(status="training_completed", duration_seconds=time.perf_counter() - started,
                      execution=collect_execution_metadata(model))
    write_json(runtime_path("run_config_file"), run_config, mode="w")
    return str(destination)


## 8. 설정 확인 후 실행

기존 결과가 있으면 중단합니다. 실행 중 학습 입력이나 정답을 조용히 자르지 않습니다.


In [ ]:
print(json.dumps(RUNTIME_PARAMS, ensure_ascii=False, indent=2))
print(json.dumps(HYPER_PARAMS, ensure_ascii=False, indent=2))


In [ ]:
adapter_path = run_training()
print("어댑터와 토크나이저 저장:", adapter_path)
